<a href="https://colab.research.google.com/github/metarun/Mechanistic-Interpretability-Exploration/blob/main/Attention_Mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Simple Self-Attention Mechanism without trainable weights**

Here I am assuming all our K - Key, Q - Query, V - Value metrics are same just for learning purpose for now


$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

$$
\text{head}_i = \text{Attention}\left(QW_i^Q,\; KW_i^K,\; VW_i^V\right)
$$

The self-attention formula is:
$$
\text{head}_i = \text{Attention}\left(QW_i^Q,\; KW_i^K,\; VW_i^V\right)
$$
This formula can be broken down into several key steps:

1.  **Query (Q), Key (K), and Value (V) Projections**: These are derived from the input embeddings. Each input token is transformed into three different vectors: a Query vector, a Key vector, and a Value vector.
    *   **Query**: Represents what we are looking for or interested in from other tokens.
    *   **Key**: Represents what each token offers as information to be matched against queries.
    *   **Value**: Represents the actual content or information of each token that will be used to construct the output.

2.  **Dot Product between Query and Keys ($QK^T$)**: This step calculates a similarity score between each Query vector and all Key vectors. It measures how much each input token (represented by its Key) is relevant to the current token (represented by its Query).

3.  **Scaling by $\sqrt{d_k}$**: The dot product scores can become very large, especially with high-dimensional vectors ($d_k$ is the dimension of the key vectors). Dividing by $\sqrt{d_k}$ helps to prevent the softmax function from having extremely small gradients, which can hinder the training process by pushing the softmax output towards a one-hot distribution (making it hard to learn small differences in relevance).

4.  **Softmax Activation**: The scaled scores are then passed through a softmax function. This normalizes the scores into a probability distribution, ensuring that all attention weights for a single query sum up to 1. These normalized weights represent how much attention should be paid to each token in the sequence when processing the current query token.

5.  **Weighted Sum of Values $$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$** Finally, these attention weights are multiplied by the Value vectors. This results in a weighted sum of the Value vectors, where tokens that are more relevant (have higher attention weights) contribute more to the final output representation. This output is a context-aware representation for each token, incorporating information from the entire sequence based on relevance.

In [ ]:
import pandas as pd

# Define the labels for the input tokens based on the comment in the 'inputs' tensor --> Below
token_labels = [
    "Your",      # x^1
    "journey",   # x^2
    "starts",    # x^3
    "with",      # x^4
    "one",       # x^5
    "step"       # x^6
]
token_labels

['Your', 'journey', 'starts', 'with', 'one', 'step']

In [ ]:
import torch


inputs = torch.tensor( # Tokens
     [[0.43, 0.15, 0.89], # Your      (x^1)
      [0.55, 0.87, 0.66], # journey   (x^2)
      [0.57, 0.85, 0.64], # starts    (x^3)
      [0.22, 0.58, 0.33], # with      (x^4)
      [0.77, 0.25, 0.10], # one       (x^5)
      [0.05, 0.80, 0.55], # step      (x^6)
    ]
)

# This is our sample input sentance " Your journey starts with one step"

In [ ]:
input_query_1 = inputs[1] # This is our current input token -- > Journey
input_query_1

tensor([0.5500, 0.8700, 0.6600])

### Here now I am trying to get ($QK^T$)

In [ ]:
# Q -- > input_query_1
# K -- > Kt

In [ ]:
Kt = inputs.T
Kt

tensor([[0.4300, 0.5500, 0.5700, 0.2200, 0.7700, 0.0500],
        [0.1500, 0.8700, 0.8500, 0.5800, 0.2500, 0.8000],
        [0.8900, 0.6600, 0.6400, 0.3300, 0.1000, 0.5500]])

In [ ]:
# Q @ Kt -- > input_query * Kt
attention_score_for_query_1 = input_query_1 @ Kt
attention_score_for_query_1

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [ ]:
# But we wand to calculate attention score for each token in the sentance hence we our Q will be entire sentance
attention_score = inputs @ Kt
attention_score

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

### I am assuming it 1 for now $${\sqrt{d_k}}$$

In [ ]:
# Now we need to take **softmax** for our attention_score

def softmax_naive(x):
  return torch.exp(x) / torch.exp(x).sum(dim=0)

In [ ]:
attention_weight_for_query_1 = softmax_naive(attention_score_for_query_1)
attention_weight_for_query_1

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [ ]:
# Use pytorch softmax
attention_weight_for_query_1 = torch.softmax(attention_score_for_query_1, dim=0)
attention_weight_for_query_1

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

**Find context vector / Atention with respect to 2nd input token - journey $$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$**

In [ ]:
# Find context vector with respect to 2nd input token - journey
#
context_vector_for_query_1 = attention_weight_for_query_1 @ inputs
context_vector_for_query_1 # This is


tensor([0.4419, 0.6515, 0.5683])

### So our token - Journey was [0.55, 0.87, 0.66] but now we have added attention weight and also gave it the context for entire sentance hence now it is transformed and is now more alingend with context of entire sentance hence now it is [0.4419, 0.6515, 0.5683]

## Lets now calculate the context vector for each token with respect to all other tokens in the sentance

* input_query == > inputs
* Key == > inputs ( For this example)
* Value = > inputs ( For this example)
* Lets calculate ($K^T$)

In [ ]:
#Lets calculate ($K^T$)
Kt = inputs.T
Kt

tensor([[0.4300, 0.5500, 0.5700, 0.2200, 0.7700, 0.0500],
        [0.1500, 0.8700, 0.8500, 0.5800, 0.2500, 0.8000],
        [0.8900, 0.6600, 0.6400, 0.3300, 0.1000, 0.5500]])

* Lets calculate ($QK^T$)

In [ ]:
QKt = inputs @ Kt
QKt

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [85]:
# Show Attention Score for all inputs with respect to other tokens
QKt_df = pd.DataFrame(QKt.numpy(), index=token_labels, columns=token_labels)

print("Attention Score (rows are Queries, columns are dimensions of the context vector):")
display(QKt_df)

Attention Score (rows are Queries, columns are dimensions of the context vector):


,Your,journey,starts,with,one,step
Your,0.9995,0.9544,0.9422,0.4753,0.4576,0.6310
journey,0.9544,1.4950,1.4754,0.8434,0.7070,1.0865
starts,0.9422,1.4754,1.4570,0.8296,0.7154,1.0605
with,0.4753,0.8434,0.8296,0.4937,0.3474,0.6565
one,0.4576,0.7070,0.7154,0.3474,0.6654,0.2935
step,0.6310,1.0865,1.0605,0.6565,0.2935,0.9450


* Lets take Softmax of ($QK^T$)

In [ ]:
SoftMaxQKt = torch.softmax(QKt, dim=1)
SoftMaxQKt

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [86]:
# Show Attention Score for all inputs with respect to other tokens
SoftMaxQKt_df = pd.DataFrame(SoftMaxQKt.numpy(), index=token_labels, columns=token_labels)

print("Attention Score (rows are Queries, columns are dimensions of the context vector):")
display(SoftMaxQKt_df)

Attention Score (rows are Queries, columns are dimensions of the context vector):


,Your,journey,starts,with,one,step
Your,0.209835,0.200581,0.198149,0.124228,0.122049,0.145158
journey,0.138548,0.237891,0.233274,0.123992,0.108182,0.158114
starts,0.139008,0.236921,0.232602,0.124204,0.110800,0.156464
with,0.143527,0.207394,0.204552,0.146192,0.126295,0.172039
one,0.152611,0.195839,0.197491,0.136687,0.187859,0.129514
step,0.138471,0.218364,0.212759,0.142048,0.098806,0.189552


Lets find the attention/context vectors for all tokens

**$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$**

In [ ]:
# Multiple SoftMaxQKt with V ==> inputs
AttentionWeights = SoftMaxQKt @ inputs
AttentionWeights

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [ ]:
# Create the DataFrame with correct dimensions
attention_df = pd.DataFrame(AttentionWeights.numpy(), index=token_labels)

print("Attention Weights (rows are Queries, columns are dimensions of the context vector):")
display(attention_df)

Attention Weights (rows are Queries, columns are dimensions of the context vector):


,0,1,2
Your,0.442059,0.593099,0.578989
journey,0.441866,0.651482,0.568309
starts,0.443128,0.649595,0.567073
with,0.430390,0.629828,0.551027
one,0.467102,0.590993,0.526597
step,0.417724,0.650323,0.564535


This DataFrame `attention_df` now shows the new, context-aware representations for each of your original tokens. For instance, the first row, labeled 'Your', now has a vector `[0.4421, 0.5931, 0.5790]`, which is its updated representation incorporating information from all other tokens in the sentence through the self-attention mechanism.

### Single step do all these

In [81]:
AttentionWeightsSingle = torch.softmax(inputs @ inputs.T, dim=1) @ inputs
AttentionWeightsSingle


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [83]:
# Create the DataFrame with correct dimensions
attention_df = pd.DataFrame(AttentionWeightsSingle.numpy(), index=token_labels)

print("Attention Weights (rows are Queries, columns are dimensions of the context vector):")
display(attention_df)

Attention Weights (rows are Queries, columns are dimensions of the context vector):


,0,1,2
Your,0.442059,0.593099,0.578989
journey,0.441866,0.651482,0.568309
starts,0.443128,0.649595,0.567073
with,0.430390,0.629828,0.551027
one,0.467102,0.590993,0.526597
step,0.417724,0.650323,0.564535
